# KanEval: Kannada LLM Evaluation Framework

This notebook demonstrates the evaluation of multiple LLMs for **Kannada text summarization** using four key metrics:

| Metric | Description |
|---|---|
| Semantic Similarity | Cosine similarity of multilingual sentence embeddings |
| Lexical Diversity | Unique token ratio in the summary |
| Avg Sentence Length | Average word count per sentence |
| Information Loss | Missing / added words vs original |

**Models compared:** GPT-4 (Model A), Gemini (Model B), DeepSeek (Model C)

## 1. Install & Import Dependencies

In [ ]:
# Uncomment to install if needed
# !pip install sentence-transformers pandas matplotlib seaborn pyyaml

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from metrics import (
    semantic_similarity,
    lexical_diversity,
    avg_sentence_length,
    information_loss
)

print("All imports successful!")

## 2. Load Configuration

In [ ]:
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

weights = config["scoring"]["weights"]
model_map = {v["label"]: v["name"] for v in config["models"].values()}

print("Scoring Weights:")
for k, v in weights.items():
    print(f"  {k}: {v}")

print("\nModel Mapping:")
for k, v in model_map.items():
    print(f"  {k} → {v}")

## 3. Sample Kannada Text & Summaries

Below is a sample original Kannada paragraph and three model-generated summaries for evaluation.

In [ ]:
original_text = """
ಕರ್ನಾಟಕವು ದಕ್ಷಿಣ ಭಾರತದ ಒಂದು ರಾಜ್ಯವಾಗಿದ್ದು, ಇದು ತನ್ನ ಶ್ರೀಮಂತ ಸಾಂಸ್ಕೃತಿಕ ಪರಂಪರೆ,
ವೈವಿಧ್ಯಮಯ ಭೂಗೋಳ ಮತ್ತು ಐತಿಹಾಸಿಕ ಮಹತ್ವಕ್ಕಾಗಿ ಪ್ರಸಿದ್ಧವಾಗಿದೆ. ಇದರ ರಾಜಧಾನಿ ಬೆಂಗಳೂರು,
ಭಾರತದ ತಂತ್ರಜ್ಞಾನ ಕೇಂದ್ರವೆಂದು ಹೆಸರುವಾಸಿಯಾಗಿದೆ. ಕನ್ನಡ ಭಾಷೆಯು ಇಲ್ಲಿನ ಅಧಿಕೃತ ಭಾಷೆಯಾಗಿದೆ.
"""

# Simulated model summaries
summaries = {
    "Model A": "ಕರ್ನಾಟಕ ದಕ್ಷಿಣ ಭಾರತದ ರಾಜ್ಯ. ರಾಜಧಾನಿ ಬೆಂಗಳೂರು ತಂತ್ರಜ್ಞಾನ ಕೇಂದ್ರ. ಕನ್ನಡ ಅಧಿಕೃತ ಭಾಷೆ.",
    "Model B": "ಕರ್ನಾಟಕ ಸಾಂಸ್ಕೃತಿಕ ಪರಂಪರೆ ಮತ್ತು ಐತಿಹಾಸಿಕ ಮಹತ್ವಕ್ಕೆ ಹೆಸರುವಾಸಿ. ಬೆಂಗಳೂರು ರಾಜಧಾನಿ.",
    "Model C": "ದಕ್ಷಿಣ ಭಾರತದ ಕರ್ನಾಟಕ ರಾಜ್ಯದ ರಾಜಧಾನಿ ಬೆಂಗಳೂರು ಭಾರತದ ತಂತ್ರಜ್ಞಾನ ಕೇಂದ್ರ."
}

print("Original text and summaries loaded.")

## 4. Run Evaluation

In [ ]:
data = []

for name, summary in summaries.items():
    sem   = semantic_similarity(original_text, summary)
    lex   = lexical_diversity(summary)
    sent  = avg_sentence_length(summary)
    missing, added = information_loss(original_text, summary)

    data.append({
        "Model":               name,
        "LLM":                 model_map[name],
        "Semantic Similarity": round(sem, 4),
        "Lexical Diversity":   round(lex, 4),
        "Avg Sentence Length": round(sent, 2),
        "Missing Info":        missing,
        "Added Info":          added
    })

df = pd.DataFrame(data)

# Weighted final score using config weights
df["Final Score"] = (
    weights["semantic_similarity"] * df["Semantic Similarity"] +
    weights["lexical_diversity"]   * df["Lexical Diversity"] +
    weights["avg_sentence_length"] * (df["Avg Sentence Length"] / df["Avg Sentence Length"].max())
).round(4)

df = df.sort_values("Final Score", ascending=False).reset_index(drop=True)
df.index = df.index + 1
df.index.name = "Rank"

df

## 5. Visualizations

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle("KanEval: LLM Evaluation Results", fontsize=15, fontweight="bold")

metrics_to_plot = [
    ("Semantic Similarity", axes[0, 0], "steelblue"),
    ("Lexical Diversity",   axes[0, 1], "mediumseagreen"),
    ("Avg Sentence Length", axes[1, 0], "darkorange"),
    ("Final Score",         axes[1, 1], "mediumpurple"),
]

for metric, ax, color in metrics_to_plot:
    bars = ax.bar(df["Model"], df[metric], color=color, edgecolor="white", linewidth=0.8)
    ax.set_title(metric, fontsize=11, fontweight="bold")
    ax.set_ylabel(metric)
    ax.set_ylim(0, df[metric].max() * 1.25)
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{bar.get_height():.4f}",
            ha="center", va="bottom", fontsize=9
        )

plt.tight_layout()
plt.savefig("kaneval_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved as kaneval_results.png")

In [ ]:
# Information Loss: Missing vs Added words
fig, ax = plt.subplots(figsize=(7, 4))
x = range(len(df))
width = 0.35

ax.bar([i - width/2 for i in x], df["Missing Info"], width, label="Missing Info", color="tomato")
ax.bar([i + width/2 for i in x], df["Added Info"],   width, label="Added Info",   color="cornflowerblue")

ax.set_xticks(list(x))
ax.set_xticklabels(df["Model"])
ax.set_title("Information Coverage Analysis", fontweight="bold")
ax.set_ylabel("Word Count")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Best Model & Export

In [ ]:
best = df.iloc[0]
print(f"🏆 Best Model: {best['Model']} ({best['LLM']})")
print(f"   Final Score:         {best['Final Score']}")
print(f"   Semantic Similarity: {best['Semantic Similarity']}")
print(f"   Lexical Diversity:   {best['Lexical Diversity']}")
print(f"   Avg Sentence Length: {best['Avg Sentence Length']}")

# Export
df.to_csv(config["export"]["default_filename"], index=True)
print(f"\nResults exported to {config['export']['default_filename']}")